# Where and when are Bigfoot reports recorded?

Two views of the **same BFRO reports dataset**, prepared for IS445. These are reports in a collected dataset, not verified sightings, a census, or measures of risk. Both charts use the same set of dated reports from 1970–2018.

Data: https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv

The date field may be approximate; patterns can reflect reporting, collection, and exposure differences. The partial 2010s decade ends in 2018. No population or observation-effort adjustment is possible here.

Use Python 3.13. Run all cells from the repository root (or from `python_notebooks/`). Requirements are listed in `python_notebooks/requirements-bfro.txt`.

In [1]:
from pathlib import Path
from io import StringIO
import hashlib
import calendar
import json
import sys
import pandas as pd
import altair as alt
import requests
from IPython.display import display

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "Altair:", alt.__version__)
DATA_URL = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"
response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()
raw = pd.read_csv(StringIO(response.text))
print("Source rows:", len(raw))
print("Source SHA256:", hashlib.sha256(response.content).hexdigest())
display(raw[["number", "date", "state", "classification"]].head())

Python: 3.13.7
pandas: 3.0.6 Altair: 5.5.0


Source rows: 4747
Source SHA256: 302c6b7cc392dfe52c9c4890bf96f2b3ccdf1911fef803a3e86d1a3540071c88


,number,date,state,classification
0,1261.0,NaN,Alaska,Class A
1,438.0,NaN,New Jersey,Class B
2,55269.0,2016-06-07,New Hampshire,Class A
3,59757.0,2018-05-25,Nebraska,Class B
4,751.0,2000-04-21,North Dakota,Class A


## Clean once; use the same records in both views

Coerce the date field to timestamps and remove records with an unparseable/missing date or a blank state. Keep 1970–2018 inclusive to focus on recent reporting while keeping complete calendar years in the supplied snapshot. Record the exclusions rather than silently dropping them. Check report identifiers for duplicates before aggregation. State names are trimmed; years, months, and decade labels are derived from the parsed dates. No report narratives, coordinates, or witness details are exported.

Zero-fill the aggregation grids so a state with no reports after a selected starting year has a true zero, and a month with no reports is distinct from a missing month. A decade with no records for a selected state is explicitly labeled as having no reports.

In [2]:
dates = pd.to_datetime(raw["date"], errors="coerce")
states = raw["state"].astype("string").str.strip()
valid = dates.notna() & states.notna() & states.ne("")
in_window = valid & dates.dt.year.between(1970, 2018)
reports = raw.loc[in_window, ["number"]].copy()
reports["state"] = states.loc[in_window]
reports["date"] = dates.loc[in_window]
assert reports["number"].notna().all()
assert not reports["number"].duplicated().any(), "Review duplicate report identifiers before aggregating."
reports["year"] = reports["date"].dt.year.astype(int)
reports["month"] = reports["date"].dt.month.astype(int)
reports["decade_start"] = (reports["year"] // 10) * 10
reports["decade"] = reports["decade_start"].astype(str) + "s"
audit = {
    "source_rows": int(len(raw)),
    "invalid_date_rows": int(dates.isna().sum()),
    "valid_date_missing_state_rows": int((dates.notna() & ~valid).sum()),
    "outside_1970_2018_rows": int((valid & ~in_window).sum()),
    "retained_rows": int(len(reports)),
    "states": int(reports.state.nunique()),
    "source_url": DATA_URL,
    "source_sha256": hashlib.sha256(response.content).hexdigest(),
}
assert audit["source_rows"] == audit["invalid_date_rows"] + audit["valid_date_missing_state_rows"] + audit["outside_1970_2018_rows"] + audit["retained_rows"]
display(pd.Series(audit).to_frame("value"))
state_names = sorted(reports["state"].unique().tolist())
years = list(range(1970, 2019))
decades = [f"{y}s" for y in range(1970, 2020, 10)]
months = list(calendar.month_abbr)[1:]
state_year_index = pd.MultiIndex.from_product([state_names, years], names=["state", "year"])
state_year = (reports.groupby(["state", "year"]).size().reindex(state_year_index, fill_value=0).rename("reports").reset_index())
assert state_year.reports.sum() == len(reports)
display(state_year.head())

,value
source_rows,4747
invalid_date_rows,950
valid_date_missing_state_rows,0
outside_1970_2018_rows,134
retained_rows,3663
states,49
source_url,https://raw.githubusercontent.com/UIUC-iSchool...
source_sha256,302c6b7cc392dfe52c9c4890bf96f2b3ccdf1911fef803...


,state,year,reports
0,Alabama,1970,0
1,Alabama,1971,0
2,Alabama,1972,0
3,Alabama,1973,0
4,Alabama,1974,0


## Plot 1 — state rankings since a chosen year

A horizontal bar chart ranks the 15 states with the most retained reports since the chosen starting year. State is nominal (`N`), and report count is quantitative (`Q`) on a zero-based axis. All bars use one teal color: color does not encode an additional variable because the position and length already express the comparison. A bound slider filters the preaggregated state/year table. Vega-Lite then sums the selected years and recomputes the ranking, breaking ties alphabetically. Tooltips give exact counts. These are raw totals, not per-capita rates.

In [3]:
start_year = alt.param(
    name="start_year", value=1970,
    bind=alt.binding_range(min=1970, max=2018, step=1, name="Include reports from: ")
)
ranked = (alt.Chart(state_year)
    .transform_filter(alt.datum.year >= start_year)
    .transform_aggregate(reports="sum(reports)", groupby=["state"])
    .transform_filter(alt.datum.reports > 0)
    .transform_window(state_rank="row_number()", sort=[alt.SortField("reports", order="descending"), alt.SortField("state", order="ascending")])
    .transform_filter(alt.datum.state_rank <= 15))
chart1 = (ranked.mark_bar(color="#176b68", cornerRadiusEnd=3)
    .encode(
        x=alt.X("reports:Q", title="Number of reports", scale=alt.Scale(zero=True), axis=alt.Axis(tickMinStep=1)),
        y=alt.Y("state:N", title=None, sort="-x"),
        tooltip=[alt.Tooltip("state:N", title="State"), alt.Tooltip("reports:Q", title="Reports", format=",")]
    ).add_params(start_year)
    .properties(width="container", height=420,
        title=alt.TitleParams(text="Where were reports recorded?", subtitle="Top 15 states · selected start year through 2018", anchor="start"))
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=12, titleFontSize=12, gridColor="#e5ece8")
    .configure_title(fontSize=18, subtitleFontSize=12, color="#193a36", subtitleColor="#5f716b"))
display(chart1)

alt.Chart(...)

## Plot 2 — the seasonal shape of reporting

A heatmap compares month-of-year distributions across five decades. Month and decade are ordered categories (`O`). The color encodes a quantitative (`Q`) within-decade share: the month's count divided by the total for that decade in the selected state. The sequential **Viridis** palette has an ordered lightness progression that suits increasing shares. Its legend rescales with the selected state, so consult the values when comparing states. Tooltips show both count and percentage. A gray row indicates a decade with no reports in the selected state.

In Python, group the cleaned records by state, decade, and month; add an all-states aggregate; and reindex every state/decade to all 12 months. Calculate decade totals and monthly shares. Normalization makes a decade's seasonal pattern readable despite different reporting volumes; it does not correct for population, reporting effort, or the shorter 2010s period.

In [4]:
monthly_counts = reports.groupby(["state", "decade", "month"]).size().rename("reports")
all_counts = reports.groupby(["decade", "month"]).size().rename("reports").reset_index()
all_counts["state"] = "All states"
combined = pd.concat([monthly_counts.reset_index(), all_counts], ignore_index=True)
monthly_index = pd.MultiIndex.from_product([["All states"] + state_names, decades, range(1, 13)], names=["state", "decade", "month"])
monthly = (combined.set_index(["state", "decade", "month"])["reports"].reindex(monthly_index, fill_value=0).reset_index())
monthly["decade_total"] = monthly.groupby(["state", "decade"])["reports"].transform("sum")
monthly["share"] = monthly["reports"].div(monthly["decade_total"].replace(0, float("nan"))).fillna(0)
monthly["share_label"] = monthly.apply(lambda r: f"{r['share']:.1%}" if r["decade_total"] else "No reports in decade", axis=1)
monthly["month_name"] = monthly["month"].map(dict(enumerate(months, start=1)))
assert monthly.loc[monthly.state.eq("All states"), "reports"].sum() == len(reports)
nonempty = monthly.groupby(["state", "decade"]).agg(total=("reports", "sum"), share=("share", "sum"))
assert nonempty.loc[nonempty.total.gt(0), "share"].sub(1).abs().max() < 1e-10
state_choice = alt.param(name="state_choice", value="All states", bind=alt.binding_select(options=["All states"] + state_names, name="Explore a state: "))
chart2 = (alt.Chart(monthly).transform_filter(alt.datum.state == state_choice)
    .mark_rect(stroke="white", strokeWidth=2, cornerRadius=3)
    .encode(
        x=alt.X("month_name:O", title="Month of recorded observation", sort=months, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("decade:O", title=None, sort=decades),
        color=alt.condition("datum.decade_total > 0", alt.Color("share:Q", title="Share of decade", scale=alt.Scale(scheme="viridis", zero=True), legend=alt.Legend(format=".0%", orient="bottom", gradientLength=250)), alt.value("#e7ebe9")),
        tooltip=[alt.Tooltip("state:N", title="State"), alt.Tooltip("decade:O", title="Decade"), alt.Tooltip("month_name:O", title="Month"), alt.Tooltip("reports:Q", title="Reports", format=","), alt.Tooltip("decade_total:Q", title="Reports in decade", format=","), alt.Tooltip("share_label:N", title="Share of decade")]
    ).add_params(state_choice)
    .properties(width="container", height=245,
        title=alt.TitleParams(text="When in the year were reports recorded?", subtitle="Monthly share within each decade · 1970–2018", anchor="start"))
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=12, titleFontSize=12, grid=False, domain=False, ticks=False)
    .configure_title(fontSize=18, subtitleFontSize=12, color="#193a36", subtitleColor="#5f716b"))
display(chart2)

alt.Chart(...)

## Export the exact specifications used by Jekyll

The JSON specifications contain aggregated data, so the deployed charts do not need Python, a server-side notebook, or access to the original CSV. The source data still comes from the URL above when rerunning the notebook. The notebook's GitHub `blob/main/...ipynb` page is linked from the project rather than the raw notebook download.

In [5]:
ROOT = Path.cwd()
if ROOT.name == "python_notebooks":
    ROOT = ROOT.parent
assert (ROOT / "_config.yml").exists(), "Run from the repository root or python_notebooks/"
JSON_DIR = ROOT / "assets" / "json"
PNG_DIR = ROOT / "assets" / "pngs"
JSON_DIR.mkdir(parents=True, exist_ok=True)
PNG_DIR.mkdir(parents=True, exist_ok=True)
chart1.save(JSON_DIR / "bfro-state-rankings.json")
chart2.save(JSON_DIR / "bfro-monthly-patterns.json")
chart1.properties(width=760).save(PNG_DIR / "bfro-state-rankings.png", scale_factor=1.5)
(JSON_DIR / "bfro-summary.json").write_text(json.dumps(audit, indent=2) + "\n")
print("Saved two Vega-Lite JSON specifications, a project thumbnail, and the cleaning summary.")
display(reports.groupby("state").size().sort_values(ascending=False).head(15).rename("reports").to_frame())

Saved two Vega-Lite JSON specifications, a project thumbnail, and the cleaning summary.


,reports
state,
Washington,485
Florida,266
Ohio,251
California,240
Illinois,198
Texas,196
Oregon,174
Michigan,144
Missouri,118
